# BRKGA Foco — Flow Shop Scheduling

Experimentos com o brkga_scs

## 0. Setup — Instalação e Imports

In [ ]:
# Instala dependencias (roda uma vez)
!pip install -q pymoo scikit-fuzzy optuna numpy pandas
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('Dependencias instaladas.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.7 MB/s eta 0:00:00
Dependencias instaladas.


In [ ]:
# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ROOT = '/content/drive/MyDrive/FlowShopScheduling'

import sys, os, time, json, csv
sys.path.insert(0, ROOT)

from pathlib import Path
from typing import List, Dict, Optional, Tuple, Callable

import numpy as np
import pandas as pd
import optuna
import json

DATA_DIR = Path(ROOT) / 'data' / 'taillard'
RESULTADOS_DIR = Path(ROOT) / 'resultados_brkga_foco'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'Resultados em: {RESULTADOS_DIR}')

ROOT: /content/drive/MyDrive/FlowShopScheduling
DATA_DIR: /content/drive/MyDrive/FlowShopScheduling/data/taillard
Resultados em: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_foco


In [ ]:
# Imports do projeto
from src.flowshop_engine import FlowShopEngine
from src.solvers.adapter import rodar_brkga, rodar_brkga_scs, SolverResult
from src.solvers.brkga_solver import (
    rodar_optuna,
    analisar_melhor_configuracao,
    construir_referencia_a_partir_da_trajetoria,
)

print('Imports concluídos.')

Imports concluídos.


## 1. Catálogo de Instâncias — Taillard

Extrai todas as instâncias com 16 jobs do diretório `data/Small/`.
Parseia o nome do arquivo: `I_{grupo}_{n_jobs}_{n_machines}_{id}.txt`.

In [ ]:
def parse_nome_instancia(nome: str) -> Dict:
    """
    Converte 'I_2_16_5_1.txt' -> {grupo: 2, n_jobs: 16, n_machines: 5, id: 1}
    """
    stem = nome.replace('.txt', '')
    partes = stem.split('_')
    return {
        'grupo': int(partes[1]),
        'n_jobs': int(partes[2]),
        'n_machines': int(partes[3]),
        'id': int(partes[4]),
        'arquivo': nome,
    }


def construir_catalogo_16x(data_dir: Path) -> pd.DataFrame:
    """
    Retorna DataFrame com todas as instancias de 16 jobs.
    """
    todos = sorted(data_dir.glob('I_*_16_*.txt'))
    registros = []
    for f in todos:
        info = parse_nome_instancia(f.name)
        info['path'] = str(f)
        registros.append(info)

    df = pd.DataFrame(registros)
    df = df.sort_values(['grupo', 'n_machines', 'id']).reset_index(drop=True)
    df['rotulo'] = df.apply(
        lambda r: f'{r["n_jobs"]}x{r["n_machines"]}_g{r["grupo"]}_id{r["id"]}', axis=1
    )
    return df

'''
catalogo = construir_catalogo_16x(DATA_DIR)
print(f'Total de instancias 16x: {len(catalogo)}')
print()
print('Distribuicao:')
for m in sorted(catalogo['n_machines'].unique()):
    qtd = len(catalogo[catalogo['n_machines'] == m])
    print(f'  16x{m}: {qtd} instancias')
print()
print('10 primeiras:')
print(catalogo.head(10).to_string(index=False))
'''


def construir_catalogo_taillard(data_dir: Path) -> pd.DataFrame:
    """
    Lê os arquivos ta*.txt, extrai as dimensões e retorna um DataFrame do catálogo.
    """
    # Encontra todos os arquivos que começam com "ta" e terminam com ".txt"
    todos = sorted(data_dir.glob('ta*.txt'))
    registros = []

    for f in todos:
        # Extrai o ID numérico (ex: "ta005.txt" -> 5)
        id_instancia = int(f.stem.replace('ta', ''))

        # Dá uma "espiada" rápida no arquivo para extrair jobs e machines da Linha 2
        with open(f, 'r') as file:
            linhas = [l.strip() for l in file if l.strip()]
            partes = linhas[1].split()
            n_jobs = int(partes[0])
            n_machines = int(partes[1])
        bks = int(partes[3]) # Já extrai o BKS para termos na tabela!

        registros.append({
            'arquivo': f.name,
            'id': id_instancia,
            'n_jobs': n_jobs,
            'n_machines': n_machines,
            'bks': bks,
            'path': str(f),
            'rotulo': f'{n_jobs}x{n_machines}_id{id_instancia}'
        })

    df = pd.DataFrame(registros)
    df = df.sort_values(['n_jobs', 'n_machines', 'id']).reset_index(drop=True)
    return df

# Aponte para a nova pasta que nosso script criou
catalogo = construir_catalogo_taillard(DATA_DIR)

print(f'Total de instâncias Taillard: {len(catalogo)}')
print()
print('Distribuição por tamanho (Jobs x Máquinas):')
# Agrupa para mostrar quantos arquivos temos de cada tamanho
distribuicao = catalogo.groupby(['n_jobs', 'n_machines']).size()
for (j, m), qtd in distribuicao.items():
    print(f'  {j}x{m}: {qtd} instâncias')

print('\n10 primeiras instâncias:')
print(catalogo.head(10).to_string(index=False))

Total de instâncias Taillard: 120

Distribuição por tamanho (Jobs x Máquinas):
  20x5: 10 instâncias
  20x10: 10 instâncias
  20x20: 10 instâncias
  50x5: 10 instâncias
  50x10: 10 instâncias
  50x20: 10 instâncias
  100x5: 10 instâncias
  100x10: 10 instâncias
  100x20: 10 instâncias
  200x10: 10 instâncias
  200x20: 10 instâncias
  500x20: 10 instâncias

10 primeiras instâncias:
  arquivo  id  n_jobs  n_machines  bks                                                              path    rotulo
ta001.txt   1      20           5 1278 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta001.txt  20x5_id1
ta002.txt   2      20           5 1359 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta002.txt  20x5_id2
ta003.txt   3      20           5 1081 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta003.txt  20x5_id3
ta004.txt   4      20           5 1293 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta004.txt  20x5_id4
ta005.txt   5      20           5 1236 /c

## 2. Configuração Global

In [ ]:
# Parametros do BRKGA (padrao fabrica)
POP_SIZE = 100
N_ELITES_DEFAULT = 20
N_OFFSPRINGS_DEFAULT = 70
N_MUTANTS_DEFAULT = 10
BIAS_DEFAULT = 0.7

# Geracoes para o experimento comparativo
MAX_EVALS = 50000

#Teste com apenas 50 geracoes

# Sementes para o experimento
N_SEEDS = 5
SEED_INICIO = 100

# Objetivo
OBJECTIVE = 'Makespan'

# Tuning
INSTANCIA_ALVO = 'I_2_16_5_1.txt'   # alvo principal
N_TRIALS_OPTUNA = 15
SEEDS_TUNING = (1, 2, 3)            # 3 seeds por trial
N_TRIALS_OPTUNA_SCS = 20  # Tuning SCS mais parametros entao tem que ter mais trials

print('Configuracao:')
print(f'  Populacao: {POP_SIZE} ({N_ELITES_DEFAULT} elites, {N_OFFSPRINGS_DEFAULT} offsprings, {N_MUTANTS_DEFAULT} mutantes)')

print(f'  Avaliacoes/exec (MAX_EVALS): {MAX_EVALS}')
print(f'  Seeds: {N_SEEDS}  ({SEED_INICIO}..{SEED_INICIO + N_SEEDS - 1})')
print(f'  Objetivo: {OBJECTIVE}')
print(f'  Instancia alvo tuning: {INSTANCIA_ALVO}')
print(f'  Trials Optuna: {N_TRIALS_OPTUNA}')


Configuracao:
  Populacao: 100 (20 elites, 70 offsprings, 10 mutantes)
  Avaliacoes/exec (MAX_EVALS): 50000
  Seeds: 5  (100..104)
  Objetivo: Makespan
  Instancia alvo tuning: I_2_16_5_1.txt
  Trials Optuna: 15


## 3. Tuning com Optuna — Instância Alvo 16×5

Roda o Optuna na instância alvo (`I_2_16_5_1.txt`)
para encontrar os melhores parâmetros: `frac_elites`, `frac_mutantes`, `bias`.
salva em json

In [ ]:
# Carrega a engine da instancia alvo -> falta conversar para decidir a melhor abordagem ...
engine_alvo = FlowShopEngine(
    *FlowShopEngine.carregar_instancia_txt(
        str(DATA_DIR / INSTANCIA_ALVO)
    )
)
print(f'Engine alvo: {engine_alvo.n_jobs} jobs, {engine_alvo.n_machines} maquinas')

In [ ]:
print('Carregando o tuning com Optuna...')
t0_tuning = time.time()

study, melhor_config = rodar_optuna(
    engine_alvo,
    pop_size=POP_SIZE,
    max_evals=MAX_EVALS,           # 100 gen durante o tuning tambem
    seeds=SEEDS_TUNING,
    n_trials=N_TRIALS_OPTUNA,
    seed_optuna=0,
)

elapsed_tuning = time.time() - t0_tuning
print(f'Tuning concluido em {elapsed_tuning:.1f}s')
print(f'Melhor custo: {study.best_value:.2f}')
print(f'Melhor config: {melhor_config}')

# Salva config em JSON
config_path = RESULTADOS_DIR / 'config_optuna.json'
with open(config_path, 'w') as f:
    json.dump(melhor_config, f, indent=2)
print(f'Config salva em: {config_path}')


## 3.1 - Optuna para o BRKGA-SCS

In [ ]:
"""
#NAO TA PRONTO AINDA
print('----Tuning Optuna para BRKGA-SCS...')
t0_scs = time.time()

def objetivo_scs(trial):
    params = {
        'pe': trial.suggest_float('pe', 0.05, 0.40),
        'pm': trial.suggest_float('pm', 0.05, 0.40),
        'rho': trial.suggest_float('rho', 0.4, 0.9),
        'z_limiar': trial.suggest_float('z_limiar', 0.5, 3.0),
        'votos_limiar': trial.suggest_int('votos_limiar', 2, 10),
        'step_reduction': trial.suggest_float('step_reduction', 0.1, 0.9),
    }
    if params['pe'] + params['pm'] > 0.80:
        raise optuna.TrialPruned()

    custos = []
    for sd in SEEDS_TUNING:
      #Alterei o n_gen passado ao rodar optuna, -> deixei fixo 100, voltar para variavel global N_GEN
        r = rodar_brkga_scs(engine_alvo, seed=sd, n_gen=100, tam_pop=POP_SIZE, **params)
        custos.append(r.best_cost)
    return float(np.mean(custos))

study_scs = optuna.create_study(direction='minimize')
study_scs.optimize(objetivo_scs, n_trials=N_TRIALS_OPTUNA_SCS)
elapsed_scs = time.time() - t0_scs

config_scs = study_scs.best_params
print(f'Tuning SCS concluido em {elapsed_scs:.1f}s')
print(f'Melhor custo: {study_scs.best_value:.2f}')
print(f'Melhor config: {config_scs}')

config_scs_path = RESULTADOS_DIR / 'config_optuna_scs.json'
with open(config_scs_path, 'w') as f:
    json.dump(config_scs, f, indent=2)
print(f'Config SCS salva em: {config_scs_path}')
"""
config_scs = {} # Mock vazio para não quebrar a célula abaixo
print('Optuna para SCS pulado temporariamente.')


## 4. Calibração das Referências Fuzzy

Usa a melhor config do Optuna para gerar as trajetórias médias
de diversidade e qualidade

In [ ]:
print('>>> Calibrando referencias fuzzy...')
t0_ref = time.time()

_, traj_div, traj_qual, resumo_calib = analisar_melhor_configuracao(
    engine_alvo,
    melhor_config,
    max_evals=MAX_EVALS,
    seeds=(10, 11, 12, 13, 14),  # 5 seeds para a calibracao
)

ref_diversidade = construir_referencia_a_partir_da_trajetoria(traj_div)
ref_qualidade = construir_referencia_a_partir_da_trajetoria(traj_qual)

elapsed_ref = time.time() - t0_ref
print(f'Calibracao concluida em {elapsed_ref:.1f}s')
print(f'Makespan medio na calibracao: {resumo_calib["makespan_medio"]:.1f}')
print(f'  Trajetoria diversidade (10 primeiros): {np.round(traj_div[:10], 4)}')
print(f'  Trajetoria qualidade (10 primeiros):   {np.round(traj_qual[:10], 4)}')


## 5. Fábrica de Variantes (closures)

Cada variante é uma callable `(engine, seed) -> SolverResult`.

In [ ]:
""" Isso é para nao ter que ficar passando os paramtros toda vez que chamar os alg...."""

def fabricar_variante_fixo_default() -> Callable:
    """BRKGA com parametros default"""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fixo_default',
            max_evals=MAX_EVALS,
            n_elites=N_ELITES_DEFAULT,
            n_offsprings=N_OFFSPRINGS_DEFAULT,
            n_mutants=N_MUTANTS_DEFAULT,
            bias=BIAS_DEFAULT,
        )
    return _run


def fabricar_variante_fixo_optuna(config: dict) -> Callable:
    """BRKGA com parametros encontrados pelo Optuna."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fixo_optuna',
            max_evals=MAX_EVALS,
            n_elites=config['n_elites'],
            n_offsprings=config['n_offsprings'],
            n_mutants=config['n_mutants'],
            bias=config['bias'],
        )
    return _run


def fabricar_variante_fuzzy(
    config: dict,
    ref_div: Callable,
    ref_qual: Callable,
) -> Callable:
    """BRKGA adaptativo com controle fuzzy, partindo da config do Optuna."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fuzzy',
            max_evals=MAX_EVALS,
            n_elites=config['n_elites'],
            n_offsprings=config['n_offsprings'],
            n_mutants=config['n_mutants'],
            bias=config['bias'],
            ref_diversidade=ref_div,
            ref_qualidade=ref_qual,
        )
    return _run


def fabricar_variante_brkga_scs() -> Callable:
    """BRKGA-SCS (Clustering Search) com parametros padrao do notebook original."""
    def _run(engine, seed):
        return rodar_brkga_scs(
            engine, seed=seed, max_evals=MAX_EVALS,
            t_param=60, # fator multiplicante para o tempo de cpu
            tam_pop=POP_SIZE, pe=0.2, pm=0.15, rho=0.7,
            z_limiar=1.5, votos_limiar=5,
            num_clusters_max=5, r_min=0.05, r_max=0.25,
            base_cooldown=5, gamma=1.5,
        )
    return _run

#DASITVADO POR ENQUANTO PQ VOU TESTAR O SCS PURO COM AS MODIFICACOES DE RAIO ADAPTATIVO e outras paradinhas
'''
def fabricar_variante_brkga_scs_optuna(config_scs: dict) -> Callable:
    """BRKGA-SCS com parametros encontrados pelo Optuna."""
    def _run(engine, seed):
        return rodar_brkga_scs(
            engine, seed=seed, max_evals=MAX_EVALS,
            tam_pop=POP_SIZE, pe=config_scs["pe"],
            pm=config_scs["pm"], rho=config_scs["rho"],
            z_limiar=config_scs["z_limiar"],
            votos_limiar=config_scs["votos_limiar"],
            step_reduction=config_scs["step_reduction"],
            raio_cluster=0.5,
        )
    return _run
'''


# Monta o dicionario de variantes
variantes = {
    #'fixo_default': fabricar_variante_fixo_default(),
    #'fixo_optuna': fabricar_variante_fixo_optuna(melhor_config),
    #'fuzzy': fabricar_variante_fuzzy(melhor_config, ref_diversidade, ref_qualidade),
    'brkga_scs': fabricar_variante_brkga_scs(),
    # 'brkga_scs_optuna': fabricar_variante_brkga_scs_optuna(config_scs), # Desabilitado temporariamente
}

print(f'Variantes disponiveis: {list(variantes.keys())}')


Variantes disponiveis: ['brkga_scs']


## 6. Loop Comparativo - Taillard


In [ ]:
def amostrar_historico_v1(history: list, max_evals: int, passo: int = 2000) -> dict:
    """Extrai o makespan nos marcos evals=passo, 2*passo, ..., max_evals."""
    amostras = {}

    # history é lista de (evals, makespan)
    # se history estiver vazio, retorne NaN
    if not history:
        for ev in range(passo, max_evals + 1, passo):
            amostras[f'mks_ev{ev}'] = float('nan')
        return amostras

    for ev in range(passo, max_evals + 1, passo):
        # Encontra o ultimo makespan reportado ANTES ou NO marco ev
        best_mks = history[0][1]
        for e, m in history:
            if e <= ev:
                best_mks = m
            else:
                break
        amostras[f'mks_ev{ev}'] = float(best_mks)
    return amostras


# Schema fixo do CSV
CAMPOS_CSV = [
    'instancia', 'grupo', 'n_jobs', 'n_machines',
    'metodo', 'seed',
    'makespan', 'tardiness', 'cfo_count', 'tempo_s',
    'lower_bound', 'bks', 'rpd',
]
# Campos de convergencia (a cada 5000 avaliacoes)
PASSO_AMOSTRA = 5000
CAMPOS_CONVERGENCIA = [f'mks_ev{e}' for e in range(PASSO_AMOSTRA, MAX_EVALS + 1, PASSO_AMOSTRA)]
CAMPOS_CSV.extend(CAMPOS_CONVERGENCIA)

#Quiser alterar o arquivo csv de comparacao é aqui essa merda
CSV_PATH = RESULTADOS_DIR / 'resultados_brkga_scs_taillard.csv'
print(f'CSV de saida: {CSV_PATH}')
print(f'Campos: {CAMPOS_CSV}')


CSV de saida: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_foco/resultados_brkga_scs_taillard.csv
Campos: ['instancia', 'grupo', 'n_jobs', 'n_machines', 'metodo', 'seed', 'makespan', 'tardiness', 'cfo_count', 'tempo_s', 'lower_bound', 'bks', 'rpd', 'mks_ev5000', 'mks_ev10000', 'mks_ev15000', 'mks_ev20000', 'mks_ev25000', 'mks_ev30000', 'mks_ev35000', 'mks_ev40000', 'mks_ev45000', 'mks_ev50000']


In [ ]:
def amostrar_historico(historico, max_evals, passo):
    amostras = {}
    for ev in range(passo, max_evals + 1, passo):
        amostras[f'mks_ev{ev}'] = float('nan')

    if not historico:
        return amostras

    alvos = list(range(passo, max_evals + 1, passo))
    idx_alvo = 0
    best_mks = historico[0][-1] # Pega sempre o último elemento do item (makespan)

    # Lendo o histórico de forma segura independente de ter 2 ou 3 colunas:
    for item in historico:
        n_evals = item[0]
        atual_mks = item[-1]

        while idx_alvo < len(alvos) and n_evals >= alvos[idx_alvo]:
            amostras[f'mks_ev{alvos[idx_alvo]}'] = float(best_mks)
            idx_alvo += 1

        best_mks = atual_mks

    while idx_alvo < len(alvos):
        amostras[f'mks_ev{alvos[idx_alvo]}'] = float(best_mks)
        idx_alvo += 1

    return amostras

In [ ]:
print('=' * 72)
print('INICIANDO LOOP COMPARATIVO')
print(f'{len(catalogo)} instancias x {len(variantes)} variantes x {N_SEEDS} seeds')
print(f'Total: {len(catalogo) * len(variantes) * N_SEEDS} execucoes')
print('=' * 72)

t0_total = time.time()

with open(DATA_DIR / 'taillard_bks.json', 'r') as f:
    bks_db = json.load(f)

csv_header_escrito = False

#for idx_inst, (_, row) in enumerate(catalogo.head(5).iterrows()):
for idx_inst, (_, row) in enumerate(catalogo.iterrows()):
    inst_label = row['rotulo']
    inst_path = row['path']
    n_jobs = row['n_jobs']
    n_mach = row['n_machines']
    #grupo = row['grupo']
    grupo = "Taillard"

    # Carrega engine uma vez por instancia
    nj, nm, pt, dd = FlowShopEngine.carregar_instancia_taillard(inst_path)
    engine = FlowShopEngine(nj, nm, pt, dd)

    for metodo, fn_variante in variantes.items():
        for seed in range(SEED_INICIO, SEED_INICIO + N_SEEDS):
            # LOG de progresso detalhado
            print(
                f'[{idx_inst + 1}/{len(catalogo)}] '
                f'Inst: {row["arquivo"]} ({n_jobs}j x {n_mach}m, g{grupo}) '
                f'| Var: {metodo} '
                f'| Seed: {seed} ...',
                end=' '
            )

            try:
                t0_exec = time.time()
                result = fn_variante(engine, seed)
                elapsed = time.time() - t0_exec

                # Monta linha do CSV
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float(result.best_cost),
                    'tardiness': float(result.tardiness),
                    'cfo_count': int(result.n_evaluations),
                    'tempo_s': round(elapsed, 3),
                    'lower_bound': float(result.lower_bound),
                }

                # Nova logica RPD
                bks_real = bks_db.get(row['arquivo'].replace('.txt', ''))
                if bks_real is not None:
                    rpd = ((result.best_cost - bks_real) / bks_real) * 100.0
                else:
                    rpd = float('nan')
                linha['bks'] = bks_real
                linha['rpd'] = round(rpd, 3)
                # Amostras de convergencia
                amostras = amostrar_historico(result.history, MAX_EVALS, PASSO_AMOSTRA)
                linha.update(amostras)

                print(f'OK  Makespan: {result.best_cost:.1f} ({elapsed:.2f}s)')

            except Exception as e:
                print(f'ERRO: {e}')
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float('nan'),
                    'tardiness': float('nan'),
                    'cfo_count': 0,
                    'tempo_s': float('nan'),
                    'lower_bound': float('nan'),
                    'bks': float('nan'),
                    'rpd': float('nan'),
                }
                for c in CAMPOS_CONVERGENCIA:
                    linha[c] = float('nan')

            # Escreve no CSV incremental (abre/fecha a cada linha = seguro)
            escrever_header = not csv_header_escrito
            with open(CSV_PATH, 'w' if escrever_header else 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=CAMPOS_CSV)
                if escrever_header:
                    writer.writeheader()
                    csv_header_escrito = True
                writer.writerow(linha)

elapsed_total = time.time() - t0_total
print('=' * 72)
print(f'LOOP CONCLUIDO em {elapsed_total:.1f}s ({elapsed_total/60:.1f}min)')
print(f'Resultados salvos em: {CSV_PATH}')


INICIANDO LOOP COMPARATIVO
120 instancias x 1 variantes x 5 seeds
Total: 600 execucoes
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 100 ... OK  Makespan: 1297.0 (3.19s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 101 ... OK  Makespan: 1297.0 (3.01s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 102 ... OK  Makespan: 1305.0 (3.01s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 103 ... OK  Makespan: 1297.0 (3.01s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 104 ... OK  Makespan: 1297.0 (3.01s)
[2/120] Inst: ta002.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 100 ... OK  Makespan: 1363.0 (3.01s)
[2/120] Inst: ta002.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 101 ... OK  Makespan: 1365.0 (3.01s)
[2/120] Inst: ta002.txt (20j x 5m, gTaillard) | Var: brkga_scs | Seed: 102 ... OK  Makespan: 1366.0 (3.01s)
[2/120] Inst: ta002.txt (20j x 5m, gTaillard) | V

## 7. Resumo Final

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Total de linhas no CSV: {len(df)}')
print()

print('=' * 120)
print('TABELA COMPARATIVA — Makespan medio por Instancia x Metodo')
print('=' * 120)

# Tabela detalhada por instancia + metodo
agg = df.groupby(['instancia', 'n_jobs', 'n_machines', 'metodo']).agg(
    makespan_medio=('makespan', 'mean'),
    desvio=('makespan', 'std'),
    melhor=('makespan', 'min'),
    pior=('makespan', 'max'),
    tempo_medio=('tempo_s', 'mean'),
    bks = ('bks', 'first'),
    rpd_medio=('rpd', 'mean'),
    lower_bound=('lower_bound', 'first'),
).reset_index()
agg['makespan_medio'] = agg['makespan_medio'].round(1)
agg['desvio'] = agg['desvio'].round(1)
agg['tempo_medio'] = agg['tempo_medio'].round(3)
agg['rpd_medio'] = agg['rpd_medio'].round(2)

print(agg.to_string(index=False))
print()

# Resumo global (agregando todas as instancias)
global_agg = df.groupby('metodo').agg(
    makespan_medio=('makespan', 'mean'),
    desvio_std=('makespan', 'std'),
    melhor_global=('makespan', 'min'),
    tempo_medio=('tempo_s', 'mean'),
    rpd_medio=('rpd', 'mean'),
).reset_index()
global_agg['makespan_medio'] = global_agg['makespan_medio'].round(1)
global_agg['desvio_std'] = global_agg['desvio_std'].round(1)
global_agg['tempo_medio'] = global_agg['tempo_medio'].round(3)
global_agg['rpd_medio'] = global_agg['rpd_medio'].round(2)

print('-' * 80)
print('RESUMO GLOBAL (120 instancias, 5 seeds)')
print('-' * 80)
print(global_agg.to_string(index=False))
print()
print(f'Arquivo salvo em: {CSV_PATH}')

Total de linhas no CSV: 600

TABELA COMPARATIVA — Makespan medio por Instancia x Metodo
instancia  n_jobs  n_machines    metodo  makespan_medio  desvio  melhor    pior  tempo_medio   bks  rpd_medio  lower_bound
ta001.txt      20           5 brkga_scs          1298.6     3.6  1297.0  1305.0        3.045  1278       1.61       1249.0
ta002.txt      20           5 brkga_scs          1365.2     1.3  1363.0  1366.0        3.008  1359       0.46       1346.0
ta003.txt      20           5 brkga_scs          1107.0    14.5  1098.0  1132.0        3.008  1081       2.41       1073.0
ta004.txt      20           5 brkga_scs          1318.8    14.3  1309.0  1344.0        3.008  1293       2.00       1268.0
ta005.txt      20           5 brkga_scs          1258.4    20.5  1235.0  1291.0        3.008  1236       1.81       1198.0
ta006.txt      20           5 brkga_scs          1210.6     7.0  1200.0  1219.0        3.008  1195       1.31       1180.0
ta007.txt      20           5 brkga_scs          12